In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
i = 5
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    i -= 1
    if i == 0: break
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/LICENSE
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/README.md


In [2]:
!uv pip install --upgrade wandb

Using Python 3.12.12 environment at: /usr
Resolved 20 packages in 362ms                                        
Prepared 6 packages in 471ms                                             
Uninstalled 6 packages in 130ms
Installed 6 packages in 30ms                                
 - gitpython==3.1.45
 + gitpython==3.1.46
 - packaging==26.0rc2
 + packaging==26.0
 - platformdirs==4.5.1
 + platformdirs==4.9.2
 - protobuf==5.29.5
 + protobuf==6.33.5
 - sentry-sdk==2.42.1
 + sentry-sdk==2.53.0
 - wandb==0.22.2
 + wandb==0.25.0


In [3]:
import os
import random
import warnings
import gc
import torch
import torch.nn as nn
import torchaudio
import multiprocessing
import wandb
import pytorch_lightning as pl
from tqdm.notebook import tqdm
from concurrent.futures import ProcessPoolExecutor
from torch.utils.data import Dataset, DataLoader
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, TQDMProgressBar
from pytorch_lightning.loggers import WandbLogger
from transformers import ASTForAudioClassification, ASTFeatureExtractor
from torchmetrics.classification import MulticlassF1Score

# Suppress audio backend warnings
warnings.filterwarnings("ignore")
pl.seed_everything(42)

2026-02-22 10:46:10.577307: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771757170.748539     109 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771757170.800047     109 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771757171.209024     109 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771757171.209061     109 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771757171.209064     109 computation_placer.cc:177] computation placer alr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Seed set to 42


42

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

# 1. Login (This will ask for your API key if not already cached)
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

CONFIG = {
    "project_name": "23f2001635-t12026",
    "experiment_name": "scratch_cnn_v1",
    "data_root": "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup",
    "save_dir": "/kaggle/working/processed_data",
    "lr": 1e-4,             # AST requires a much lower learning rate
    "batch_size": 16,        # Transformers are large; reduce batch size to prevent OOM
    "accumulate_grad": 4,   # Simulates a batch size of 32 (8 * 4)
    "num_epochs": 100,      # Set high; Early Stopping will halt it automatically
    "patience": 5,          # Halt if val_f1 doesn't improve for 5 epochs
    "num_classes": 10,
    "sample_rate": 16000,   # MUST be 16000 for HuggingFace AST
    "duration": 30,
    "train_samples": 3000,
    "val_samples": 500,
    "num_workers": multiprocessing.cpu_count(), # Maximize CPU threads for data loading,
    "mode": "train", # gen or train
    "train_samples": 3400,
    "val_samples": 600
}

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: harshitsahu01 (harshitsahu01-iitm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
# --- HELPER WORKER ---
def _generate_single_sample(args):
    """Runs on a single CPU core to generate one mixed audio sample."""
    # We no longer need split_name since we are generating all at once
    stems_root, noise_files, genres, genre_songs, label_map, sample_rate, duration = args
    target_len = sample_rate * duration
    
    genre = random.choice(genres)
    label = label_map[genre]
    mashup = torch.zeros(target_len)
    
    # Mix 4 random stems from the same genre
    if genre_songs.get(genre):
        songs = random.choices(genre_songs[genre], k=4)
        stem_names = ['drums.wav', 'bass.wav', 'vocals.wav', 'other.wav']
        
        for i, stem_name in enumerate(stem_names):
            path = os.path.join(songs[i], stem_name)
            try:
                waveform, sr = torchaudio.load(path)
                if sr != sample_rate:
                    waveform = torchaudio.transforms.Resample(sr, sample_rate)(waveform)
                if waveform.shape[0] > 1:
                    waveform = torch.mean(waveform, dim=0, keepdim=True)
                stem = waveform.squeeze(0)
                
                if stem.size(0) < target_len:
                    stem = torch.nn.functional.pad(stem, (0, target_len - stem.size(0)))
                else:
                    stem = stem[:target_len]
                mashup += stem
            except Exception:
                pass 

    # Add ESC-50 Noise (Applying to all generated samples for consistent distribution)
    if noise_files and random.random() > 0.2:
        try:
            n_path = random.choice(noise_files)
            noise, sr = torchaudio.load(n_path)
            if sr != sample_rate:
                noise = torchaudio.transforms.Resample(sr, sample_rate)(noise)
            if noise.shape[0] > 1:
                noise = torch.mean(noise, dim=0, keepdim=True)
            noise = noise.squeeze(0)
            
            if noise.size(0) < target_len:
                noise = torch.nn.functional.pad(noise, (0, target_len - noise.size(0)))
            else:
                noise = noise[:target_len]
            mashup += noise * random.uniform(0.1, 0.4)
        except Exception:
            pass

    # Normalize audio to prevent clipping
    max_val = torch.max(torch.abs(mashup))
    if max_val > 0:
        mashup = mashup / max_val
        
    return mashup, label

# --- MAIN GENERATION FUNCTION ---
def generate_train_mashups(config):
    """Main function to orchestrate multi-core data generation and split into train/val."""
    print(f"--- Starting CPU Multicore Generation ---")
    os.makedirs(config["save_dir"], exist_ok=True)
    
    stems_root = os.path.join(config["data_root"], 'genres_stems')
    noise_dir = os.path.join(config["data_root"], 'ESC-50-master', 'audio')
    
    genres = sorted([d for d in os.listdir(stems_root) if os.path.isdir(os.path.join(stems_root, d))])
    label_map = {g: i for i, g in enumerate(genres)}
    
    genre_songs = {g: [os.path.join(stems_root, g, s) for s in os.listdir(os.path.join(stems_root, g)) 
                       if os.path.isdir(os.path.join(stems_root, g, s))] for g in genres}
                       
    noise_files = [os.path.join(noise_dir, f) for f in os.listdir(noise_dir) if f.endswith('.wav')] if os.path.exists(noise_dir) else []

    num_cores = config["num_workers"]
    total_samples = config["train_samples"] + config["val_samples"]
    
    print(f"Utilizing {num_cores} CPU cores to generate {total_samples} samples in a single run...")

    # Pack arguments for the single massive loop
    task_args = [(stems_root, noise_files, genres, genre_songs, label_map, 
                  config["sample_rate"], config["duration"]) for _ in range(total_samples)]
    
    all_audio = []
    all_labels = []
    
    with ProcessPoolExecutor(max_workers=num_cores) as executor:
        results = list(tqdm(executor.map(_generate_single_sample, task_args), total=total_samples, desc="Generating All Data"))
        
    for audio, label in results:
        all_audio.append(audio)
        all_labels.append(label)

    print("\nGeneration complete. Slicing data into Train and Validation sets...")

    # 1. Slice and save the Training Data
    train_audio = torch.stack(all_audio[:config["train_samples"]])
    train_labels = torch.tensor(all_labels[:config["train_samples"]])
    
    torch.save(train_audio, os.path.join(config["save_dir"], 'train_audio.pt'))
    torch.save(train_labels, os.path.join(config["save_dir"], 'train_labels.pt'))
    print(f"Saved Train data: {train_audio.shape}")
    
    # Free memory immediately
    del train_audio, train_labels

    # 2. Slice and save the Validation Data
    val_audio = torch.stack(all_audio[config["train_samples"]:])
    val_labels = torch.tensor(all_labels[config["train_samples"]:])

    torch.save(val_audio, os.path.join(config["save_dir"], 'val_audio.pt'))
    torch.save(val_labels, os.path.join(config["save_dir"], 'val_labels.pt'))
    print(f"Saved Validation data: {val_audio.shape}")

    # Free memory immediately
    del val_audio, val_labels

    # 3. Final Garbage Collection
    del all_audio, all_labels, results
    gc.collect()
    print("Memory successfully cleared and ready for GPU training.")

In [8]:
# --- LIGHTNING DATASET & DATAMODULE ---
class ASTDataset(Dataset):
    def __init__(self, split_name, config):
        self.data = torch.load(os.path.join(config["save_dir"], f'{split_name}_audio.pt'))
        self.labels = torch.load(os.path.join(config["save_dir"], f'{split_name}_labels.pt'))
        
        # HuggingFace extractor converts raw audio to the specific spectrograms AST expects
        self.feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
        self.sr = config["sample_rate"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        audio = self.data[idx].numpy()
        label = self.labels[idx]
        
        # Process audio to AST Spectrogram
        inputs = self.feature_extractor(audio, sampling_rate=self.sr, return_tensors="pt")
        input_values = inputs["input_values"].squeeze(0) # Remove batch dim added by extractor
        
        return input_values, label

class ASTDataModule(pl.LightningDataModule):
    def __init__(self, config):
        super().__init__()
        self.config = config

    def setup(self, stage=None):
        self.train_ds = ASTDataset('train', self.config)
        self.val_ds = ASTDataset('val', self.config)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.config["batch_size"], 
                          shuffle=True, num_workers=self.config["num_workers"], pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.config["batch_size"], 
                          shuffle=False, num_workers=self.config["num_workers"], pin_memory=True)

# --- LIGHTNING MODEL ---
class ASTLightningModule(pl.LightningModule):
    def __init__(self, config):
        super().__init__()
        self.save_hyperparameters(config)
        self.lr = config["lr"]
        
        # Load Pretrained Audio Spectrogram Transformer
        self.model = ASTForAudioClassification.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593",
            num_labels=config["num_classes"],
            ignore_mismatched_sizes=True # Rewires the classification head for 10 classes
        )
        
        self.loss_fn = nn.CrossEntropyLoss()
        self.f1_metric = MulticlassF1Score(num_classes=config["num_classes"], average='macro')

    def forward(self, input_values):
        outputs = self.model(input_values=input_values)
        return outputs.logits

    def training_step(self, batch, batch_idx):
        input_values, labels = batch
        logits = self(input_values)
        loss = self.loss_fn(logits, labels)
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_values, labels = batch
        logits = self(input_values)
        loss = self.loss_fn(logits, labels)
        
        preds = torch.argmax(logits, dim=1)
        f1 = self.f1_metric(preds, labels)
        
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_f1', f1, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-3)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=2
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_f1"
            }
        }

# --- MAIN TRAINING FUNCTION ---
def train_pipeline(config):
    print(f"\n--- Starting GPU Training Pipeline ---")
    
    wandb.login()
    wandb_logger = WandbLogger(project=config["project_name"], name=config["experiment_name"], config=config)

    data_module = ASTDataModule(config)
    model = ASTLightningModule(config)

    checkpoint_callback = ModelCheckpoint(
        monitor='val_f1',
        mode='max',
        save_top_k=1,
        filename='best-ast-{epoch:02d}-{val_f1:.3f}',
        verbose=True
    )

    early_stop_callback = EarlyStopping(
        monitor='val_f1',
        min_delta=0.005,
        patience=config["patience"],
        verbose=True,
        mode='max'
    )

    trainer = pl.Trainer(
        max_epochs=config["num_epochs"],
        logger=wandb_logger,
        accelerator="auto",
        devices="auto",
        accumulate_grad_batches=config["accumulate_grad"],
        precision="16-mixed", # Uses half-precision to speed up transformers significantly
        callbacks=[checkpoint_callback, early_stop_callback, TQDMProgressBar(leave=True)],
        log_every_n_steps=10
    )

    trainer.fit(model, datamodule=data_module)
    
    print(f"\nTraining Complete! Best model saved at: {checkpoint_callback.best_model_path}")
    print(f"Best model validation F1 score: {checkpoint_callback.best_model_score}")
    wandb.finish()

In [9]:
if CONFIG['mode'] == 'gen':
    # 1. Generate the datasets using all CPU cores
    generate_train_mashups(CONFIG)
elif CONFIG['mode'] == 'train':
    # 2. Train the AST model on the GPU
    train_pipeline(CONFIG)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.
Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- Starting GPU Training Pipeline ---


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ ASTForAudioClassification │ 86.2 M │ eval  │     0 │
│ 1 │ loss_fn   │ CrossEntropyLoss          │      0 │ train │     0 │
│ 2 │ f1_metric │ MulticlassF1Score         │      0 │ train │     0 │
└───┴───────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 86.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 86.2 M                                                                                               
Total estimated model params size (MB): 344                                                                        
Modules in train mode: 2                                                                                           
Modules in eval mode: 216                                                                                          
Total FLOPs: 0

...

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_f1 did not improve in the last 5 records. Best score: 0.982. Signaling Trainer to stop.
Epoch 11, global step 648: 'val_f1' reached 0.98693 (best 0.98693), saving model to './23f2001635-t12026/n47m4bku/checkpoints/best-ast-epoch=11-val_f1=0.987.ckpt' as top 1



Training Complete! Best model saved at: ./23f2001635-t12026/n47m4bku/checkpoints/best-ast-epoch=11-val_f1=0.987.ckpt
Best model validation F1 score: 0.9869348406791687


epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇█████
train_loss_epoch,█▃▂▁▁▁▁▁▁▂▁▁
train_loss_step,█▃▂▂▂▃▁▁▁▁▁▁▄▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▁▁▁▁▁
trainer/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
val_f1,▅▄▅▇▇▇█▇▇▁▇█
val_loss,▄▄▃▂▂▁▁▁▂█▂▁
epoch,11
train_loss_epoch,0.00276
train_loss_step,0.00215
trainer/global_step,647
val_f1,0.98693


In [11]:
!ls /kaggle/working/23f2001635-t12026/n47m4bku/checkpoints/best-ast-epoch=11-val_f1=0.987.ckpt

'/kaggle/working/23f2001635-t12026/n47m4bku/checkpoints/best-ast-epoch=11-val_f1=0.987.ckpt'
